# Matched Gemma FV follow-up
Enable HF_TOKEN, two T4 GPUs, and internet, then Save & Run All. Runs construction-only FV engineering pilots, followed by two tasks × three seeds × two checkpoints (12 cells). Full arms are gated on correctness and measured runtime, never favorable accuracy. Results include head means, AIE maps, selected heads/layers, random controls, and held-out predictions. See execution_protocol.md in outputs.

In [ ]:
import os,json,time,zlib,base64,sys,hashlib,datetime,importlib.metadata,traceback
from pathlib import Path
os.environ['HF_HOME']='/tmp/hf-cache'
# Preserve the environment validated by the completed bridge notebook.
import subprocess
subprocess.run([sys.executable,'-m','pip','install','--quiet','transformers==5.0.0','accelerate==1.13.0'],check=True)
import torch
from kaggle_secrets import UserSecretsClient
files=json.loads(zlib.decompress(base64.b64decode('...')))
OUT=Path('/kaggle/working');ROOT=OUT/'source'
for name,content in files.items():
 p=ROOT/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(content)
sys.path.insert(0,str(ROOT/'src'));sys.path.insert(0,str(ROOT/'scripts'))
from fvtv import kaggle_backend as backend,kaggle_fv,eval_icl,tv,fv,tasks,heldout
import run_heldout,run_sentiment_mapping
from types import SimpleNamespace
from run_heldout import write_json
backend.TOKEN=UserSecretsClient().get_secret('HF_TOKEN')
backend.REVISIONS=json.loads(files['revisions.json']);backend.OUTPUT=OUT
backend.DEADLINE=min(time.time()+11.5*3600,datetime.datetime.fromisoformat('2026-09-08T20:00:00+02:00').timestamp())
assert torch.cuda.device_count()==2, 'Expected two T4 GPUs'
for m in [run_heldout,run_sentiment_mapping]:m.load_model=backend.load_model
eval_icl.predict_top1=backend.predict_top1
tv.extract_theta_all_layers=backend.extract_theta_all_layers
tv.patch_theta=backend.patch_theta
fv.verify_arch=backend.verify_arch
fv.compute_mean_head_activations=kaggle_fv.compute_mean_head_activations
fv.compute_aie=kaggle_fv.compute_aie
fv.grab_out_proj_params=kaggle_fv.grab_out_proj_params
fv.inject_fv=kaggle_fv.inject_fv
execution={'started':time.time(),'dtype':'float16','batch_size':4,'revisions':backend.REVISIONS,'deadline':backend.DEADLINE,'versions':{p:importlib.metadata.version(p) for p in ['torch','transformers','accelerate']},'source_hashes':{name:hashlib.sha256(content.encode()).hexdigest() for name,content in files.items()},'models':{}}
write_json(OUT/'execution.json',execution)

for model in ['gemma-2-9b','gemma-2-9b-it']:
 state=execution['models'][model]={'stage':'loading','complete':False}
 write_json(OUT/'execution.json',execution)
 try:
  m=backend.load_model('google/'+model)
  construction=heldout.partition(tasks.load_task('antonym'))['construction']
  state['stage']='fv-engineering-pilot';write_json(OUT/'execution.json',execution)
  state['pilot']=kaggle_fv.engineering_pilot(m,construction)
  estimate=1.5*6*10*state['pilot']['aie_trial_seconds']+1800
  state['estimated_required_seconds']=estimate
  if backend.DEADLINE-time.time()<estimate:
   raise TimeoutError('Insufficient time for the fixed six-cell FV arm')
  del m
  state['stage']='fv-full';write_json(OUT/'execution.json',execution)
  run_heldout.run(SimpleNamespace(model=model,method='fv',tasks='antonym,country-capital',seeds='100,101,102',out=str(OUT/'results/fv'/model/'fv')))
  state.update(stage='complete',complete=True,finished=time.time())
 except Exception as exc:
  state.update(stage='failed',error_type=type(exc).__name__)
  # Source frame locations aid diagnosis without printing credentials or local variables.
  state['error_frames']=[{'file':Path(f.filename).name,'line':f.lineno,'function':f.name} for f in traceback.extract_tb(exc.__traceback__)]
  print(model,'STOPPED',type(exc).__name__,flush=True)
 m=None
 write_json(OUT/'execution.json',execution)
print('EXPERIMENT FINISHED',json.dumps(execution['models']),flush=True)
